## EIF on Time Based Data 

This notebook documents the building of an EIF model to identify outliers. 

In [7]:
%env mlflow_tracking_username = dan
%env mlflow_tracking_password = DanLarson

env: mlflow_tracking_username=dan
env: mlflow_tracking_password=DanLarson


In [1]:
import h2o
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 25.391-b13, mixed mode)
  Starting server from C:\Users\dan.larson\AppData\Local\Programs\Python\Python311\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\DAN~1.LAR\AppData\Local\Temp\4\tmp7ygpbisi
  JVM stdout: C:\Users\DAN~1.LAR\AppData\Local\Temp\4\tmp7ygpbisi\h2o_dan_larson_started_from_python.out
  JVM stderr: C:\Users\DAN~1.LAR\AppData\Local\Temp\4\tmp7ygpbisi\h2o_dan_larson_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,05 secs
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.44.0.2
H2O_cluster_version_age:,6 months and 28 days
H2O_cluster_name:,H2O_from_python_dan_larson_t6ge0w
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,6.993 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [2]:
from mlflow import MlflowClient
from pprint import pprint
import mlflow.h2o



2024/06/05 11:44:08 INFO mlflow.tracking.fluent: Experiment with name 'Extended Isolation Forest Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/dan.larson/Documents/MLAR/notebooks/2-beta/mlruns/801646859632375458', creation_time=1717602248375, experiment_id='801646859632375458', last_update_time=1717602248375, lifecycle_stage='active', name='Extended Isolation Forest Experiment', tags={}>

In [11]:
features = ['login_to_recieve_ms', 
            'recd_to_sample_ms', 
            'recd_to_started_ms', 
            'recd_to_completed_ms', 
            'started_to_completed_ms', 
            'recd_to_revewed_ms', 
            'started_to_revewed_ms', 
            'completed_to_revewed_ms', 
            'login_to_changed_ms', 
            'recd_to_changed_ms', 
            'started_to_changed_ms', 
            'completed_to_changed_ms', 
            'reviewed_to_changed_ms', 
            'login_date_compare', 
            'recd_date_compare', 
            'DATE_STARTED_compare', 
            'date_completed_compare', 
            'date_reviewed_compare']

In [20]:

import duckdb as db
time_df = db.sql("select * from read_parquet('C:/MLAR_DATA/T0001/FS_turnaround_time_table/TIME_BASED_*.parquet')").to_df()
labelled_data = db.sql("select SOURCE_DB, lower(audit_table_name) as audit_table_name, audit_table_key, max(case when severity in ('1','2','3','4','5') then 1 else 0 end) as anomaly from read_parquet('C:/MLAR_DATA/T0001/labelled_*.parquet') group by 1,2,3").to_df()
time_with_label = time_df.merge(labelled_data,left_on=['table_key','table_name','SOURCE_DB'] ,right_on=['audit_table_key','audit_table_name','SOURCE_DB'],how='left')
time_with_label['anomaly'] = time_with_label['anomaly'].fillna(0)

time_with_label = time_with_label.fillna(-99)


In [21]:
time_with_label.head()

,table_key,table_name,status,changed_on,login_date,sampled_date,recd_date,date_started,date_completed,date_reviewed,...,date_reviewed_audit_log,login_date_compare,recd_date_compare,DATE_STARTED_compare,date_completed_compare,date_reviewed_compare,SOURCE_DB,audit_table_name,audit_table_key,anomaly
0,19307,sample,A,2023-01-31 12:35:26,2022-01-27 15:07:38,2022-01-31 15:55:58,2022-02-09 15:30:08,2022-02-09 15:31:23,2022-02-09 15:31:23,2023-01-31 12:35:02,...,2023-01-31 12:35:02,True,True,True,True,True,X5,-99,-99.0,0.0
1,19316,sample,A,2023-01-31 12:35:26,2022-01-27 15:07:45,2022-01-31 15:55:58,2022-02-01 14:53:39,2022-02-07 13:49:55,2022-02-07 13:49:55,2023-01-31 12:35:02,...,2023-01-31 12:35:02,True,True,True,True,True,X5,-99,-99.0,0.0
2,37791,sample,U,2022-10-24 00:00:08,2022-10-24 00:00:07,-99,-99,-99,-99,-99,...,-99,True,False,False,False,False,X5,-99,-99.0,0.0
3,39268,sample,U,2022-12-07 00:00:09,2022-12-07 00:00:08,-99,-99,-99,-99,-99,...,-99,True,False,False,False,False,X5,-99,-99.0,0.0
4,38580,sample,A,2023-02-02 10:29:02,2022-11-14 15:53:01,2022-11-17 15:45:41,2022-11-21 13:50:17,2022-11-21 14:47:28,2022-11-21 14:47:28,2023-02-02 10:28:39,...,2023-02-02 10:28:39,True,True,True,True,True,X5,-99,-99.0,0.0


In [8]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")
# Set the experiment name (if it doesn't exist, it will be created)
mlflow.set_experiment("Extended Isolation Forest Experiment")

2024/06/05 11:50:40 INFO mlflow.tracking.fluent: Experiment with name 'Extended Isolation Forest Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Users/dan.larson/mlruns/4', creation_time=1717602641076, experiment_id='4', last_update_time=1717602641076, lifecycle_stage='active', name='Extended Isolation Forest Experiment', tags={}>

In [22]:
import h2o
from h2o.estimators import H2OIsolationForestEstimator
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score


# Initialize H2O server
h2o.init()

# Load the dataset
data = h2o.H2OFrame(time_with_label)

labels = data['anomaly']
features_data = data[features]

# Define model parameters
ntrees = 100
sample_size = 256
extension_level = 1
seed = 1234

# Initialize the model
eif_model = H2OIsolationForestEstimator(
    ntrees=ntrees,
    sample_size=sample_size,
    #extension_level=extension_level,
    seed=seed
)

# Start an MLflow run
with mlflow.start_run():
    # Log model parameters
    mlflow.log_param("ntrees", ntrees)
    mlflow.log_param("sample_size", sample_size)
    mlflow.log_param("extension_level", extension_level)
    mlflow.log_param("seed", seed)
    
    # Train the model
    eif_model.train(training_frame=features_data)
    
    # Predict anomalies
    anomalies = eif_model.predict(features_data)
    
    # Combine the scores with the original data
    data_with_anomalies = features_data.cbind(anomalies)
    data_with_anomalies = data_with_anomalies.cbind(labels)
    
    # Evaluate the model
    y_true = labels.as_data_frame().values.flatten()
    y_pred = anomalies['predict'].as_data_frame().values.flatten()
    
    # Convert predicted scores to binary labels (0: normal, 1: anomaly)
    threshold = 0.5  # You may need to adjust this threshold based on your data
    y_pred_binary = (y_pred > threshold).astype(int)
    
    # Calculate evaluation metrics
    report = classification_report(y_true, y_pred_binary)
    confusion = confusion_matrix(y_true, y_pred_binary)
    
    # Log metrics
    mlflow.log_metric("precision", precision_score(y_true, y_pred_binary))
    mlflow.log_metric("recall", recall_score(y_true, y_pred_binary))
    mlflow.log_metric("f1_score", f1_score(y_true, y_pred_binary))
    
    # Print the evaluation metrics
    print(report)
    print(confusion)
    
    # Log the model
    mlflow.h2o.log_model(eif_model, "model")
    
    # Optionally, log the data with anomalies
    output_path = "anomalies_output.csv"
    h2o.export_file(data_with_anomalies, path=output_path, force=True)
    mlflow.log_artifact(output_path)

# Shutdown H2O server
h2o.shutdown()

Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,8 mins 01 secs
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.44.0.2
H2O_cluster_version_age:,6 months and 28 days
H2O_cluster_name:,H2O_from_python_dan_larson_yj4ws2
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,6.805 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
isolationforest Model Build progress: |██████████████████████████████████████████| (done) 100%
isolationforest prediction progress: |███████████████████████████████████████████| (done) 100%
              precision    recall  f1-score   support

           0       0.98      0.99      0.99    523565
           1       0.07      0.06      0.06      9527

    accuracy                           0.97    533092
   macro avg       0.53      0.52      0.52    533092
weighted avg       0.97      0.97      0.97    533092

[[516821   6744]
 [  8981    546]]
Export File progress: |██████████████████████████████████████████████████████████| (done) 100%


C:\Users\dan.larson\AppData\Local\Temp\4\ipykernel_7588\2675586999.py:77: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown()


H2O session _sid_addf closed.


In [28]:
anomalies

predict,mean_length
0.723005,3.92
0.666667,4.16
0.0915493,6.61
0.0915493,6.61
0.201878,6.14
0.0915493,6.61
0.753521,3.79
0.173709,6.26
0.176056,6.25
0.0915493,6.61
